## The State of Tax Justice: Estimate misalignment
- Author: Alison Schultz, based on Javier Garcia Bernado's work
- Created: 4 August 2023
- Last updated: 4 August 2023

**Description**
- This notebook is the third out of three notebooks to estimate the tax losses caused by profit shifting by multinational enterprises (MNEs). The analysis used the misalignment method based on the country-by-country reports (CbCR) published by the OECD.
- This notebook estimates profit misalignment based on different formulas. It uses the dataset "data/final/cbcr_main.csv" (for the estimation with imputed values) or the dataset "data/final/cbcr_main_noimputation_allsubgroupsonly.csv" (for the estimation without imputed values). 

**Outline**
- TO DO

**To dos before running this notebook**
1. Run the notebooks 1_clean and 2_imupte_missings. Note the requirements of these notebooks. 

In [ ]:
import pandas as pd
import numpy as np
import tjn_tools
from config_aug import *
from sklearn.impute import SimpleImputer

Read in relevant data

In [ ]:
cbcr_main = pd.read_csv(f'{data_intermediate}/cbcr_main_no_imputation_allsubgroupsonly_2023.csv')
cbcr_main.rename(columns={'profit_loss_before_income_tax_corrected': 'reported_profit'}, inplace=True)
cbcr_main = cbcr_main[~cbcr_main['iso_partner'].isin(aggregated_country_groups)] # Drop country groups CHECK HOW MUCH WE LOSE HERE
# Impute wages where missing
imputer = SimpleImputer(strategy='median')
# Apply imputer on 'wage_monthly' column
cbcr_main['wage_monthly'] = imputer.fit_transform(cbcr_main[['wage_monthly']])
cbcr_main['payroll'] = cbcr_main['n_employees'] * cbcr_main['wage_monthly'] * 12 
CORRECT BELOW
cbcr_main['cit'] = cbcr_main['cit'].fillna(0)
cbcr_main['etr_average_corrected'] = cbcr_main['etr_average_corrected'].fillna(0)
cbcr_main = cbcr_main[~cbcr_main['partner_jurisdiction'].isin(aggregated_country_groups)]

In [ ]:
cbcr_main.loc[
    (cbcr_main['iso_parent'] == 'NLD') & (cbcr_main['iso_partner'] == 'NLD') & (cbcr_main['year'] == 2016)  & (cbcr_main['reported_profit'] > 0), 
    'reported_profit'
] *= 15601/22133
cbcr_main.loc[
    (cbcr_main['iso_parent'] == 'NLD') & (cbcr_main['iso_partner'] == 'NLD') & (cbcr_main['year'] == 2017) & (cbcr_main['reported_profit'] > 0), 
    'reported_profit'
] *= 20991/31008
cbcr_main.loc[
    (cbcr_main['iso_parent'] == 'NLD') & (cbcr_main['iso_partner'] == 'NLD') & (cbcr_main['year'] == 2018) & (cbcr_main['reported_profit'] > 0), 
    'reported_profit'
] *= 14617/22190
cbcr_main.loc[
    (cbcr_main['iso_parent'] == 'NLD') & (cbcr_main['iso_partner'] == 'NLD') & (cbcr_main['year'] == 2019) & (cbcr_main['reported_profit'] > 0), 
    'reported_profit'
] *= 20961/25503
cbcr_main.loc[
    (cbcr_main['iso_parent'] == 'NLD') & (cbcr_main['iso_partner'] == 'NLD') & (cbcr_main['year'] == 2020) & (cbcr_main['reported_profit'] > 0), 
    'reported_profit'
] *= 20294/24088

## 1. Define misalignment

In [ ]:
# MAY NOT BE APPLIED ON DUPLICATE VALUES, IMPUTED DATA MUST HAVE THE SAME ROWS AS THE ORIGINAL CBCR DATA
# DELETE AGGREGATED GROUPS; ONLY LOOK AT EU
def calculate_misalignment(cbcr_data,
                           formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",
                                         'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip'],
                           weights=[.5,0,0,.5, 0, 0, 0, 0],
                           profit_var="reported_profit",
                           etr_max = .15):
                           
    # share of profit cannot be negative but minimum zero --> set negative profits to zero to calculate shares
    # However, keep the original profit variables for summing up all profit later
    cbcr_data['profit_var_pos'] = cbcr_data[profit_var]
    cbcr_data.loc[cbcr_data[profit_var] < 0,'profit_var_pos'] = 0
    cbcr_data['share_profit'] = cbcr_data['profit_var_pos'] / cbcr_data.groupby('iso_parent')['profit_var_pos'].transform('sum')

    
    actual_weights = []
    actual_variables = []
    for i, var in enumerate(formula_vars):
        if var is None:
            if weights[i] != 0:
                raise(Exception("Weight of None variable is nonzero"))
        else:
            if weights[i] > 0:
                actual_variables.append("share_{}".format(var))
                actual_weights.append(weights[i])
                cbcr_data.loc[cbcr_data[var] < 0, var] = 0  # set economic activity measure to zero if negative
                cbcr_data["share_{}".format(var)] = cbcr_data[var] / cbcr_data.groupby('iso_parent')[var].transform('sum')

    cbcr_data["share_economy_partner_of_parent"] = (cbcr_data.loc[:, actual_variables] * actual_weights).sum(1, min_count=len(actual_weights))
    cbcr_data.loc[(cbcr_data["share_economy_partner_of_parent"] == 0) & (cbcr_data[profit_var] > 0), "share_economy_partner_of_parent"] = 0.01 # Set economic activity to 1% if there is no economic activity at all
    # Make sure that fractions sum up to 1
    cbcr_data["share_economy_partner_of_parent"] = cbcr_data["share_economy_partner_of_parent"] / cbcr_data.groupby('iso_parent')["share_economy_partner_of_parent"].transform('sum')

    cbcr_data["theoretical_profit"] = cbcr_data["share_economy_partner_of_parent"] * cbcr_data.groupby('iso_parent')[profit_var].transform('sum')
 
    cbcr_data["misaligned_profit"] = cbcr_data[profit_var] - cbcr_data["theoretical_profit"]
    cbcr_data.loc[((cbcr_data["misaligned_profit"] > 0)  & (cbcr_data["etr_average_corrected"] > etr_max)), "misaligned_profit"] = 0
    total_negative_misalignment = cbcr_data.loc[cbcr_data["misaligned_profit"] < 0, "misaligned_profit"].sum()
    total_positive_misalignment = cbcr_data.loc[cbcr_data["misaligned_profit"] > 0, "misaligned_profit"].sum()
    factor = - total_positive_misalignment / total_negative_misalignment
    cbcr_data.loc[cbcr_data["misaligned_profit"] < 0, "misaligned_profit"] *= factor

    return cbcr_data

 reduce Dutch numbers by new correction

Netherlands analysis

In [ ]:
# ['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

for year in range(first_year, first_year + n_years):
    print(f"Total profit shifted in USD mn {year}")
    
    misalignment_year = cbcr_main[cbcr_main['year'] == year].copy()  # Make a copy to avoid warnings
    misalignment_year = calculate_misalignment(misalignment_year, etr_max = .15, weights = [.5,0,0,.5,0,0,0,0])
    
    country_results_year = misalignment_year.groupby(['partner_jurisdiction','etr_average_corrected','cit']).agg(
    sum_negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
    sum_positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
    sum_theoretical_profit=('theoretical_profit', 'sum'),
    sum_reported_profit=('reported_profit', 'sum')
    ).reset_index()

    # report everything in million and scale by profits that actually go to low-tax jurisdictions
    country_results_year['sum_negative_misalignment'] = - country_results_year['sum_negative_misalignment'] / 1e6
    country_results_year['sum_positive_misalignment'] = country_results_year['sum_positive_misalignment'] / 1e6
    country_results_year['sum_theoretical_profit'] = country_results_year['sum_theoretical_profit'] / 1e6
    country_results_year['sum_reported_profit'] = country_results_year['sum_reported_profit'] / 1e6

    sum_positive_misalignment = country_results_year['sum_positive_misalignment'].sum()
    sum_negative_misalignment = country_results_year['sum_negative_misalignment'].sum()
    print(f"Year: {year}, Positive misalignment: {sum_positive_misalignment}, Negative misalignment: {sum_negative_misalignment}")
        
    country_results_year['net_misalignment'] = country_results_year['sum_negative_misalignment'] - country_results_year['sum_positive_misalignment']
    country_results_year['revenue_loss_incurred'] = country_results_year['sum_negative_misalignment'] * country_results_year['cit']
    
    sum_revenue_loss = country_results_year['revenue_loss_incurred'].sum()
    country_results_year['revenue_loss_inflicted_pct'] = country_results_year['sum_positive_misalignment']/sum_positive_misalignment*100
    country_results_year['revenue_loss_inflicted'] = country_results_year['revenue_loss_inflicted_pct'] * sum_revenue_loss/100
    sum_revenue_loss_caused = country_results_year['revenue_loss_inflicted'].sum()

    print(f"Year: {year}, Revenue loss incurred: {sum_revenue_loss}, Revenue loss inflicted: {sum_revenue_loss_caused} ")
   
    country_results_year = country_results_year[['partner_jurisdiction','sum_negative_misalignment',
       'sum_positive_misalignment','net_misalignment', 'revenue_loss_incurred','revenue_loss_inflicted','revenue_loss_inflicted_pct','etr_average_corrected','cit']]
    country_results_year = country_results_year.sort_values(by='partner_jurisdiction')
    country_results_year.to_csv(f'{output_tables}/Netherlands_sotj_{year}.csv')

## BEFIT estimates: no imputed values, only multinationals that are active in the EU

In [ ]:
cbcr_eu = cbcr_main[cbcr_main['eu27'] == 1]
cbcr_eu = cbcr_eu[cbcr_eu['n_employees'].notna()]
cbcr_eu = cbcr_eu[cbcr_eu['reported_profit'].notna()]
cbcr_eu["holding_or_managing_ip"].fillna(0, inplace=True)

In [ ]:
# ['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

for year in range(first_year, first_year + n_years):
    print(f"Total profit shifted in USD mn {year}")
    
    misalignment_year = cbcr_eu[cbcr_eu['year'] == year].copy()  # Make a copy to avoid warnings
    misalignment_year = calculate_misalignment(misalignment_year, etr_max = 1, weights = [1/3,1/3,1/3,0,0,0,0,0])
    
    country_results_year = misalignment_year.groupby(['iso_partner', 'partner_jurisdiction','etr_average_corrected','tax_revenue_current_usd']).agg(
    sum_negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
    sum_positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
    sum_theoretical_profit=('theoretical_profit', 'sum'),
    sum_reported_profit=('reported_profit', 'sum')
    ).reset_index()

    # report everything in million
    country_results_year['sum_negative_misalignment'] = - country_results_year['sum_negative_misalignment'] / 1e6
    country_results_year['sum_positive_misalignment'] = country_results_year['sum_positive_misalignment'] / 1e6
    country_results_year['sum_theoretical_profit'] = country_results_year['sum_theoretical_profit'] / 1e6
    country_results_year['sum_reported_profit'] = country_results_year['sum_reported_profit'] / 1e6

    sum_positive_misalignment = country_results_year['sum_positive_misalignment'].sum()
    sum_negative_misalignment = country_results_year['sum_negative_misalignment'].sum()
    
    print(sum_positive_misalignment, sum_negative_misalignment)

    country_results_year['net_misalignment'] = country_results_year['sum_negative_misalignment'] - country_results_year['sum_positive_misalignment']
    country_results_year['tax_revenue'] = country_results_year['net_misalignment'] * country_results_year['cit']
    # Calculate potential additional revenue from minimum tax
    min_tax_binding = country_results_year['etr_average_corrected'] < 0.15

    country_results_year['additional_revenue_through_minimum_tax'] = np.where(
            min_tax_binding, country_results_year['sum_theoretical_profit'] * 0.15 - country_results_year['sum_theoretical_profit'] * country_results_year['etr_average_corrected']
            ,0)
    country_results_year['tax_revenue_incl_mintax'] = country_results_year['tax_revenue'] + country_results_year['additional_revenue_through_minimum_tax']
    country_results_year['tax_revenue_befit_pct_total'] = country_results_year['tax_revenue'] * 1e6 / country_results_year['tax_revenue_current_usd'] # multiply by 1e6 as the latter is in USD
    country_results_year['tax_revenue_befit_mintax_pct_total'] = country_results_year['tax_revenue_incl_mintax'] * 1e6 / country_results_year['tax_revenue_current_usd']

    tax_effect = country_results_year['tax_revenue'].sum()
    tax_effect_incl_mintax = country_results_year['tax_revenue_incl_mintax'].sum()
    country_results_year['tax_revenue_befit_pct_total_eu'] = tax_effect * 1e6 / country_results_year['tax_revenue_current_usd'].sum()
    country_results_year['tax_revenue_befit_mintax_pct_total_eu'] = tax_effect_incl_mintax * 1e6 / country_results_year['tax_revenue_current_usd'].sum()
    print(f"Year: {year}, Additional tax revenues: {tax_effect}")
    print(f"Year: {year}, Additional tax revenues incl. minimum tax: {tax_effect_incl_mintax}")

    country_results_year = country_results_year[['partner_jurisdiction','sum_negative_misalignment',
       'sum_positive_misalignment','net_misalignment', 'tax_revenue',
       'tax_revenue_befit_pct_total', 'additional_revenue_through_minimum_tax', 'tax_revenue_incl_mintax', 
       'tax_revenue_befit_mintax_pct_total', 'tax_revenue_befit_pct_total_eu',
       'tax_revenue_befit_mintax_pct_total_eu','etr_average_corrected',]]
    country_results_year = country_results_year.sort_values(by='partner_jurisdiction')
    country_results_year.to_csv(f'{output_tables}/Netherlands_3factor_{year}.csv')

In [ ]:
# Create dictionaries to store results
sum_positive_misalignment_dict = {}
sum_negative_misalignment_dict = {}

for year in range(first_year, first_year + n_years):
    print(f"Total profit shifted in USD bn {year}")
    
    misalignment_year = cbcr_main[cbcr_main['year'] == year].copy()  # Make a copy to avoid warnings
    misalignment_year = calculate_misalignment(misalignment_year)
    
    country_results_year = misalignment_year.groupby(['iso_partner', 'partner_jurisdiction']).agg(
        sum_negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
        sum_positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
        sum_theoretical_profit=('misaligned_profit', lambda x: x[x > 0].sum()),
        sum_reported_profit=('misaligned_profit', lambda x: x[x > 0].sum())
    ).reset_index()
    
    country_results_year['sum_negative_misalignment'] = country_results_year['sum_negative_misalignment'] / 1e9
    country_results_year['sum_positive_misalignment'] = country_results_year['sum_positive_misalignment'] / 1e9
        
    sum_positive_misalignment = country_results_year['sum_positive_misalignment'].sum()
    sum_negative_misalignment = country_results_year['sum_negative_misalignment'].sum()
    
    sum_positive_misalignment_dict[year] = sum_positive_misalignment
    sum_negative_misalignment_dict[year] = sum_negative_misalignment
    
    print(sum_positive_misalignment, sum_negative_misalignment)

In [ ]:
def calculate_misalignment_javier(cbcr_data,
                           weights=[1/2,0,0,1/2,0,0,0,0],
                           profit_var="reported_profit",
                           formula_variables = ['n_employees',"unrelated_party_revenues","tangible_assets_except_cash","payroll", 
                                                'stated_capital','total_revenues','related_party_revenues','holding_or_managing_ip'],
                          carve_out=None,
                          carve_out_perc=0,
                          thres_mis=0,
                          max_etr=0.15,
                          remove_misalignment=[]):
    """
    Calculate misalignment as a function of sales, employees, capital and wages
    
    In:
    cbcr_data = data
    pi/rev/emp/cap/wages_var = variables with the profits/revenue/employees/capital/wages (None to discard)
    weights = weights of each of the variables. 
    
    Out:
    
    """
    
    #Make sure it is okay
    assert sum(weights)==1,"Weights need to sum up to 1"
    
    variables_keep = []
    weights_keep = []
    for i,var in enumerate(formula_variables):
        if var is None:
            if weights[i] != 0:
                raise(Exception("Weight of None variable is nonzero"))
        else:
            if weights[i] > 0:
                variables_keep.append("share_{}".format(var))
                weights_keep.append(weights[i])
                cbcr_data.loc[cbcr_data[var] < 0, var] = 0
                cbcr_data["share_{}".format(var)] = cbcr_data[var]/(cbcr_data[var].sum()) 
                #cbcr_data["share_{}".format(var)] = cbcr_data[var].div(cbcr_data[var].sum())  
    
    #Calculate the share of the economy
    cbcr_data["share_economy"] = (cbcr_data.loc[:,variables_keep]*weights_keep).sum(1,min_count=len(weights_keep))
    cbcr_data.loc[cbcr_data["share_economy"] == 0,cbcr_data["share_economy"]] = 0.01 # Set economic activity to 1% if there is no economic activity at all
    
    #Create carve out 
    if (carve_out is not None) and (carve_out_perc > 0):
        cbcr_data[profit_var] = cbcr_data[profit_var]-carve_out_perc*cbcr_data[carve_out]
            # set negative profits to zero
            # cbcr_data.loc[cbcr_data[profit_var]<0,profit_var] = 0
    
    #cbcr_data["share_profit"] = cbcr_data[profit_var].div(cbcr_data[profit_var].sum())
    cbcr_data.loc[cbcr_data[profit_var] < 0 , profit_var] = 0
    cbcr_data["share_profit"] = cbcr_data[profit_var]/(cbcr_data[profit_var].sum())
    #Adjustment for countries where we don't want to implement the misalignment method (in 2018 this is Saudi Arabia, because of their huge reported domestic profits of Aramco). For these countries, we just take the share of the economic activity instead of the share of profit.
    for country in remove_misalignment:
        cbcr_data.loc[(cbcr_data['iso_parent'] == country) & (cbcr_data["iso_partner"] == country), "share_profit"
                      ] = cbcr_data.loc[(cbcr_data['iso_parent'] == country) & (cbcr_data["iso_partner"] == country), "share_economy"]
    
    cbcr_data["share_profit"] = cbcr_data["share_profit"]/cbcr_data["share_profit"].sum()
        # set negative profits to zero
    
    #Theoretical profits
    cbcr_data["theoretical_profit"] = cbcr_data[profit_var].sum()*cbcr_data["share_economy"]
    
    #Misalignment
    cbcr_data["misaligned_profit"] = cbcr_data[profit_var]-cbcr_data["theoretical_profit"]

    #OVer threshold (x% of the profits can be misaligned)
    cbcr_data.loc[(cbcr_data["misaligned_profit"]>0)&(cbcr_data["misaligned_profit"]<cbcr_data[profit_var]*thres_mis),"misaligned_profit"] = 0
    
    #Threshold adjustment
    #cbcr_data.loc[(cbcr_data["misaligned_profit"]>0)&(cbcr_data["etr_average_corrected"]>max_etr)&(cbcr_data["iso_parent"]!=cbcr_data["iso_partner"]),"misaligned_profit"] = 0
    cbcr_data.loc[(cbcr_data["misaligned_profit"]>0)&(cbcr_data["etr_average_corrected"]>max_etr),"misaligned_profit"] = 0

    #Correct for discrepancies
    x = cbcr_data["misaligned_profit"]

    positive_misalignment = x[x>0].sum()
    negative_misalignment = -x[x<0].sum()
#   print("Profit shifted {:2.1f} adjusted to {:2.1f}".format(previous_pr/1E9,pos_pr/1E9))
    cbcr_data.loc[cbcr_data["misaligned_profit"]<0,"misaligned_profit"] *= positive_misalignment/negative_misalignment

    return cbcr_data


In [ ]:
for year in range(first_year, first_year + n_years):
    misalignment_year = cbcr_main[cbcr_main['year'] == year].copy()  # Filter data for the specific year and create a copy

    m_javier = calculate_misalignment_javier(misalignment_year)
    
    
    country_results_javier = m_javier.groupby(['iso_partner','partner_jurisdiction']).agg(
        sum_negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
        sum_positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum())
    ).reset_index()
    country_results_javier['sum_negative_misalignment'] = country_results_javier['sum_negative_misalignment'] / 1e9
    country_results_javier['sum_positive_misalignment'] = country_results_javier['sum_positive_misalignment'] / 1e9
    
    sum_positive_misalignment = country_results_javier['sum_positive_misalignment'].sum()
    sum_negative_misalignment = country_results_javier['sum_negative_misalignment'].sum()
    
    print(f"Year: {year}, Profit shifted: {sum_positive_misalignment}, {sum_negative_misalignment}")


In [ ]:
for year in range(first_year, first_year + n_years):
    misalignment_year = cbcr_eu[cbcr_eu['year'] == year].copy()  # Filter data for the specific year and create a copy

    m_javier = calculate_misalignment_javier(misalignment_year, max_etr = .15, weights = [1/2,0,0,1/2,0,0,0,0])
    
    
    country_results_javier = m_javier.groupby(['iso_partner','partner_jurisdiction','cit','tax_revenue_current_usd']).agg(
        sum_negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
        sum_positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum())
    ).reset_index()
    country_results_javier['sum_negative_misalignment'] = country_results_javier['sum_negative_misalignment'] / 1e6
    country_results_javier['sum_positive_misalignment'] = country_results_javier['sum_positive_misalignment'] / 1e6
    
    sum_positive_misalignment = country_results_javier['sum_positive_misalignment'].sum()
    sum_negative_misalignment = country_results_javier['sum_negative_misalignment'].sum()
    
    print(f"Year: {year}, Profit shifted: {sum_positive_misalignment}, {sum_negative_misalignment}")

    country_results_javier['net_misalignment'] = country_results_javier['sum_positive_misalignment'] + country_results_javier['sum_negative_misalignment']
    country_results_javier['tax_revenues'] = - country_results_javier['net_misalignment'] * country_results_javier['etr_average_corrected']
    country_results_javier['tax_revenue_pct_total'] = country_results_javier['tax_revenues'] * 1e6 / country_results_javier['tax_revenue_current_usd']
    country_results_javier['tax_revenue_pct_total_eu'] = tax_effect * 1e6 / country_results_javier['tax_revenue_current_usd'].sum()

    tax_effect = country_results_javier['tax_revenues'].sum()
    print(f"Year: {year}, Additional tax revenues: {tax_effect}")

    country_results_javier.to_csv(f'{output_tables}/BEFIT_javier_{year}.csv')

Estimates based on bootstrapped sample with imputed values

In [ ]:
STOP

In [ ]:
# use ETR average but substitute it to ETR foreign when ETR average is unavailable
cols_to_import = ['iso_parent_x', 'iso_partner', 'year', 'unrelated_party_revenues', 'wage_monthly',
                   'n_employees', 'tangible_assets_except_cash','profit_loss_before_income_tax_corrected',
                   'etr_average_corrected', 'etr_average_corrected', 'eu27','n_rep']
cbcr_main_bootstrapped = pd.read_csv(f'{data_final}/cbcr_main.csv', usecols = cols_to_import)
cbcr_main_bootstrapped = cbcr_main_bootstrapped[cbcr_main_bootstrapped['n_rep']<500]

cbcr_main_bootstrapped.rename(columns={'iso_parent_x':'iso_parent', 'profit_loss_before_income_tax_corrected': 'reported_profit'}, inplace=True)
cbcr_main_bootstrapped = cbcr_main_bootstrapped[~cbcr_main_bootstrapped['iso_partner'].isin(country_groups)] # Drop country groups CHECK HOW MUCH WE LOSE HERE

cbcr_main_bootstrapped['wage_monthly'] = imputer.fit_transform(cbcr_main_bootstrapped[['wage_monthly']])
cbcr_main_bootstrapped['payroll'] = cbcr_main_bootstrapped['n_employees'] * cbcr_main_bootstrapped['wage_monthly'] * 12 

In [ ]:
# Convert objects to category if they have less than 50% unique values
for col in cbcr_main_bootstrapped.columns:
    if cbcr_main_bootstrapped[col].dtype == 'object':
        num_unique_values = len(cbcr_main_bootstrapped[col].unique())
        num_total_values = len(cbcr_main_bootstrapped[col])
        if num_unique_values / num_total_values < 0.5:
            cbcr_main_bootstrapped[col] = cbcr_main_bootstrapped[col].astype('category')
# Convert integers
df_int = cbcr_main_bootstrapped.select_dtypes(include=['int'])
converted_int = df_int.apply(pd.to_numeric, downcast='unsigned')
cbcr_main_bootstrapped[converted_int.columns] = converted_int
# Convert float types
df_float = cbcr_main_bootstrapped.select_dtypes(include=['float'])
converted_float = df_float.apply(pd.to_numeric, downcast='float')
cbcr_main_bootstrapped[converted_float.columns] = converted_float

In [ ]:
all_replications = []
sum_positive_misalignment_dict = {}
sum_negative_misalignment_dict = {}

for year in range(first_year, first_year + n_years-1):
    for (rep, year), data in cbcr_main_bootstrapped.groupby(["n_rep", "year"]):
        misalignment = calculate_misalignment(data)
        country_results = misalignment.groupby(['iso_partner']).agg(
            sum_negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
            sum_positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum())
        ).reset_index()
        country_results['sum_negative_misalignment'] = country_results['sum_negative_misalignment'] / 1e9
        country_results['sum_positive_misalignment'] = country_results['sum_positive_misalignment'] / 1e9
        country_results['year'] = year
        all_replications.append(country_results)
        
        sum_positive_misalignment = country_results['sum_positive_misalignment'].sum()
        sum_negative_misalignment = country_results['sum_negative_misalignment'].sum()
        sum_positive_misalignment_dict[(rep, year)] = sum_positive_misalignment
        sum_negative_misalignment_dict[(rep, year)] = sum_negative_misalignment
        
        print(f"Year: {year}, Replication sample: {rep}, Profit shifted: {sum_positive_misalignment}, {sum_negative_misalignment}")

# Concatenate the all_replications list of DataFrames
df_all_replications = pd.concat(all_replications, ignore_index=True)

In [ ]:
# Calculate summary statistics
country_results_bootstrapped = df_all_replications.groupby(['iso_partner', 'year']).agg(
    median_positive_misalignment=('sum_positive_misalignment', 'median'),
    median_negative_misalignment=('sum_negative_misalignment', 'median'),
    avg_positive_misalignment=('sum_positive_misalignment', 'mean'),
    avg_negative_misalignment=('sum_negative_misalignment', 'mean'),
    sd_positive_misalignment=('sum_positive_misalignment', 'std'),
    sd_negative_misalignment=('sum_negative_misalignment', 'std'),
    percentile_2_5_positive_misalignment=('sum_positive_misalignment', lambda x: x.quantile(0.025)),
    percentile_2_5_negative_misalignment=('sum_negative_misalignment', lambda x: x.quantile(0.025)),
    percentile_97_5_positive_misalignment=('sum_positive_misalignment', lambda x: x.quantile(0.975)),
    percentile_97_5_negative_misalignment=('sum_negative_misalignment', lambda x: x.quantile(0.975))
).reset_index()

# Merge relevant variables to the dataset based on 'iso_partner' and 'year'
country_results_bootstrapped = country_results_bootstrapped.merge(
    cbcr_main[['iso_partner', 'year', 'gdp_current_usd', 'gvt_health_expenditure', 'population', 'cit', 'etr_average_corrected']],
    on=['iso_partner', 'year'],
    how='left'
)

In [ ]:
all_replications_javier = []
sum_positive_misalignment_dict_javier = {}
sum_negative_misalignment_dict_javier = {}

# Only use this loop to iterate over unique combinations of replications and years
for (rep, year), data in cbcr_main_bootstrapped.groupby(["n_rep", "year"]):
    misalignment_javier = calculate_misalignment_javier(data)
    country_results_javier = misalignment_javier.groupby(['iso_partner']).agg(
        sum_negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
        sum_positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum())
    ).reset_index()
    country_results_javier['sum_negative_misalignment'] = country_results_javier['sum_negative_misalignment'] / 1e9
    country_results_javier['sum_positive_misalignment'] = country_results_javier['sum_positive_misalignment'] / 1e9
    country_results_javier['year'] = year
    all_replications_javier.append(country_results_javier)
    
    sum_positive_misalignment_javier = country_results_javier['sum_positive_misalignment'].sum()
    sum_negative_misalignment_javier = country_results_javier['sum_negative_misalignment'].sum()
    sum_positive_misalignment_dict_javier[(rep, year)] = sum_positive_misalignment_javier
    sum_negative_misalignment_dict_javier[(rep, year)] = sum_negative_misalignment_javier
    
    print(f"Year: {year}, Replication sample: {rep}, Profit shifted: {sum_positive_misalignment_javier}, {sum_negative_misalignment_javier}")

# Concatenate the all_replications list of DataFrames
df_all_replications_javier = pd.concat(all_replications_javier, ignore_index=True)

In [ ]:
# Calculate summary statistics
country_results_bootstrapped_javier = df_all_replications_javier.groupby(['iso_partner', 'year']).agg(
    median_positive_misalignment=('sum_positive_misalignment', 'median'),
    median_negative_misalignment=('sum_negative_misalignment', 'median'),
    avg_positive_misalignment=('sum_positive_misalignment', 'mean'),
    avg_negative_misalignment=('sum_negative_misalignment', 'mean'),
    sd_positive_misalignment=('sum_positive_misalignment', 'std'),
    sd_negative_misalignment=('sum_negative_misalignment', 'std'),
    percentile_2_5_positive_misalignment=('sum_positive_misalignment', lambda x: x.quantile(0.025)),
    percentile_2_5_negative_misalignment=('sum_negative_misalignment', lambda x: x.quantile(0.025)),
    percentile_97_5_positive_misalignment=('sum_positive_misalignment', lambda x: x.quantile(0.975)),
    percentile_97_5_negative_misalignment=('sum_negative_misalignment', lambda x: x.quantile(0.975))
).reset_index()

# Merge relevant variables to the dataset based on 'iso_partner' and 'year'
country_results_bootstrapped_javier = country_results_bootstrapped_javier.merge(
    cbcr_main[['iso_partner', 'year', 'gdp_current_usd', 'gvt_health_expenditure', 'population', 'cit', 'etr_average_corrected']],
    on=['iso_partner', 'year'],
    how='left'
)